## This notebook will align your muscimol injections to the CCF. It will also create tools for correlating ccf-aligned injection sites with various behavior metrics from dynamic routing performance and trials tables.

In [ ]:
# to add deps:
# !uv add npc_sessions --optional dr -q

In [1]:
!uv sync --extra dr -q
# restart kernel after running!

In [2]:
import os
import polars as pl
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import datetime
import zoneinfo
import npc_lims
from dynamic_routing_analysis import data_utils, spike_utils

## Import and Clean Neuroglancer Injection Data

In [3]:
# Extract date from label (format: region_hemisphere_MM-DD-YYYY) and create session_id
# Convert MM-DD-YYYY to YYYY-MM-DD format for session_id

neuroglancer_annotations_df = pl.read_parquet(r"Z:\vayle\neuroglancer_injections_coords\neuroglancer_injections_coords.parquet")

neuroglancer_annotations_df = neuroglancer_annotations_df.with_columns(
    # Extract date portion after last underscore: MM-DD-YYYY
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 1).str.zfill(2).alias("month"),
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 2).str.zfill(2).alias("day"),
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 3).alias("year"),
).with_columns(
    # Format as YYYY-MM-DD
    (pl.col("year") + "-" + pl.col("month") + "-" + pl.col("day")).alias("date")
).with_columns(
    (pl.col("mouse_id") + "_" + pl.col("date")).alias("session_id")
).drop(["month", "day", "year"]).with_columns(
    # Round up AP, ML, DV to integers
    pl.col("AP").ceil().cast(pl.Int64),
    pl.col("ML").ceil().cast(pl.Int64),
    pl.col("DV").ceil().cast(pl.Int64),
    # Remove date from label (e.g., "ORBvl_LH_07-30-2025" -> "ORBvl_LH")
    pl.col("label").str.replace(r"_\d{1,2}-\d{1,2}-\d{4}$", "").alias("target_brain_region"),
    # Remove list brackets from description (e.g., "['text']" -> "text")
    pl.col("description").str.replace_all(r"[\[\]']", ""),
).drop("label")

neuroglancer_annotations_df

mouse_id,description,AP,DV,ML,date,session_id,target_brain_region
str,str,i64,i64,i64,str,str,str
"""767405""","""45 nL nonconjugated muscimol""",216,359,701,"""2025-05-20""","""767405_2025-05-20""","""ORBvl_LH"""
"""767405""","""45 nL nonconjugated muscimol""",217,393,471,"""2025-05-20""","""767405_2025-05-20""","""ORBvl_RH"""
"""767405""","""45 nL nonconjugated muscimol""",220,357,697,"""2025-05-29""","""767405_2025-05-29""","""ORBvl_LH"""
"""767405""","""45 nL nonconjugated muscimol""",219,394,471,"""2025-05-29""","""767405_2025-05-29""","""ORBvl_RH"""
"""767405""","""45 nL nonconjugated muscimol""",494,354,774,"""2025-05-23""","""767405_2025-05-23""","""DLS_LH"""
…,…,…,…,…,…,…,…
"""807743""","""45 nL nonconjugated muscimol""",294,398,424,"""2025-11-14""","""807743_2025-11-14""","""AId_RH"""
"""807743""","""360 nL conjugated muscimol""",289,346,635,"""2025-11-04""","""807743_2025-11-04""","""ORBm_LH"""
"""807743""","""360 nL conjugated muscimol""",303,337,563,"""2025-11-04""","""807743_2025-11-04""","""ORBm_RH"""


In [8]:
print(neuroglancer_annotations_df)

shape: (44, 8)
┌──────────┬───────────────┬─────┬─────┬─────┬────────────┬────────────────────┬───────────────────┐
│ mouse_id ┆ description   ┆ AP  ┆ DV  ┆ ML  ┆ date       ┆ session_id         ┆ target_brain_regi │
│ ---      ┆ ---           ┆ --- ┆ --- ┆ --- ┆ ---        ┆ ---                ┆ on                │
│ str      ┆ str           ┆ i64 ┆ i64 ┆ i64 ┆ str        ┆ str                ┆ ---               │
│          ┆               ┆     ┆     ┆     ┆            ┆                    ┆ str               │
╞══════════╪═══════════════╪═════╪═════╪═════╪════════════╪════════════════════╪═══════════════════╡
│ 767405   ┆ 45 nL         ┆ 216 ┆ 359 ┆ 701 ┆ 2025-05-20 ┆ 767405_2025-05-20  ┆ ORBvl_LH          │
│          ┆ nonconjugated ┆     ┆     ┆     ┆            ┆                    ┆                   │
│          ┆ muscimol      ┆     ┆     ┆     ┆            ┆                    ┆                   │
│ 767405   ┆ 45 nL         ┆ 217 ┆ 393 ┆ 471 ┆ 2025-05-20 ┆ 767405_2025-05-2

In [4]:
%store neuroglancer_annotations_df

Stored 'neuroglancer_annotations_df' (DataFrame)


## Import and Clean VS200 Slide Scanner Injection Data

In [22]:
# Import VS200 slide scanner .xlsx file with Pinpoint injection coordinates and descriptions
# Skip first row (use second row as headers), then drop first column
vs200_annotations_df = pl.read_excel(
    r"Z:\Vayle\Muscimol\VS200_slide scanner_muscimol_injection_coords.xlsx")

vs200_annotations_df = vs200_annotations_df.drop(vs200_annotations_df.columns[0])

Could not determine dtype for column 0, falling back to string


In [23]:
# Convert Pinpoint coordinates to CCF (10um resolution)
# Original coords are in μm, bregma-relative (bregma = 0,0,0)
# Take absolute value, divide by 10, round up (ceil)

# Get columns containing AP, DV, or ML
coord_cols = [col for col in vs200_annotations_df.columns if any(x in col for x in ["AP", "DV", "ML"])]
# print("Coordinate columns:", coord_cols)

vs200_annotations_df = vs200_annotations_df.with_columns([
    (pl.col(col).cast(pl.Float64, strict=False).abs() / 10).ceil().cast(pl.Int64).alias(col)
    for col in coord_cols
])


In [24]:
# Reshape vs200 from wide to long format to match neuroglancer structure
# Each injection (1-4, LH/RH) becomes a separate row

injection_rows = []

for row in vs200_annotations_df.iter_rows(named=True):
    mouse_id = str(row["mouseID"])
    
    # Iterate through injection numbers and hemispheres
    for inj_num in range(1, 5):
        for hemi in ["LH", "RH"]:
            # Get column names for this injection
            label_col = f"injection {inj_num} {hemi} label"
            ap_col = f"AP_{inj_num}_{hemi}"
            ml_col = f"ML_{inj_num}_{hemi}"
            dv_col = f"DV_{inj_num}_{hemi}"
            
            # Check if columns exist
            if label_col not in row or ap_col not in row:
                continue
                
            label = row.get(label_col)
            ap = row.get(ap_col)
            ml = row.get(ml_col)
            dv = row.get(dv_col)
            
            # Skip if label is null/empty or coordinates are null
            if label is None or ap is None or label == "died post-injection":
                continue
            
            injection_rows.append({
                "mouse_id": mouse_id,
                "label": label,
                "AP": ap,
                "ML": ml,
                "DV": dv,
                "description": row.get("description_7", "")  # Use last description column
            })

# Create long-format dataframe
vs200_annotations_df = pl.DataFrame(injection_rows)

# Now apply same transformations as neuroglancer
vs200_annotations_df = vs200_annotations_df.with_columns(
    # Extract date portion after last underscore: MM-DD-YYYY
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 1).str.zfill(2).alias("month"),
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 2).str.zfill(2).alias("day"),
    pl.col("label").str.extract(r"(\d{1,2})-(\d{1,2})-(\d{4})$", 3).alias("year"),
).with_columns(
    # Format as YYYY-MM-DD
    (pl.col("year") + "-" + pl.col("month") + "-" + pl.col("day")).alias("date")
).with_columns(
    (pl.col("mouse_id") + "_" + pl.col("date")).alias("session_id")
).drop(["month", "day", "year"]).with_columns(
    # Round up AP, ML, DV to integers (already integers from earlier conversion)
    pl.col("AP").cast(pl.Int64),
    pl.col("ML").cast(pl.Int64),
    pl.col("DV").cast(pl.Int64),
    # Remove date from label (e.g., "ORBvl_LH_07-30-2025" -> "ORBvl_LH")
    pl.col("label").str.replace(r"_\d{1,2}-\d{1,2}-\d{4}$", "").alias("target_brain_region"),
).drop("label")

vs200_annotations_df

mouse_id,AP,ML,DV,description,date,session_id,target_brain_region
str,i64,i64,i64,str,str,str,str
"""823577""",306,513,151,"""45 nL nonconugated muscimol""","""2026-03-10""","""823577_2026-03-10""","""ORBm_RH"""
"""823577""",471,725,101,"""45 nL nonconugated muscimol""","""2026-03-13""","""823577_2026-03-13""","""Ant. Str_LH"""
"""823577""",470,385,111,"""45 nL nonconugated muscimol""","""2026-03-13""","""823577_2026-03-13""","""Ant. Str_RH"""
"""823577""",282,593,189,"""45 nL nonconugated muscimol""","""2026-03-17""","""823577_2026-03-17""","""ORBm_LH"""
"""823577""",267,523,176,"""45 nL nonconugated muscimol""","""2026-03-17""","""823577_2026-03-17""","""ORBm_RH"""
…,…,…,…,…,…,…,…
"""856846""",503,396,91,"""45 nL nonconugated muscimol""","""2026-06-05""","""856846_2026-06-05""","""Ant. Str_RH"""
"""856846""",305,596,174,"""45 nL nonconugated muscimol""","""2026-06-09""","""856846_2026-06-09""","""ORBm_LH"""
"""856846""",295,536,168,"""45 nL nonconugated muscimol""","""2026-06-09""","""856846_2026-06-09""","""ORBm_RH"""


In [25]:
print(vs200_annotations_df)

shape: (31, 8)
┌──────────┬─────┬─────┬─────┬──────────────┬────────────┬────────────────────┬────────────────────┐
│ mouse_id ┆ AP  ┆ ML  ┆ DV  ┆ description  ┆ date       ┆ session_id         ┆ target_brain_regio │
│ ---      ┆ --- ┆ --- ┆ --- ┆ ---          ┆ ---        ┆ ---                ┆ n                  │
│ str      ┆ i64 ┆ i64 ┆ i64 ┆ str          ┆ str        ┆ str                ┆ ---                │
│          ┆     ┆     ┆     ┆              ┆            ┆                    ┆ str                │
╞══════════╪═════╪═════╪═════╪══════════════╪════════════╪════════════════════╪════════════════════╡
│ 823577   ┆ 306 ┆ 513 ┆ 151 ┆ 45 nL        ┆ 2026-03-10 ┆ 823577_2026-03-10  ┆ ORBm_RH            │
│          ┆     ┆     ┆     ┆ nonconugated ┆            ┆                    ┆                    │
│          ┆     ┆     ┆     ┆ muscimol     ┆            ┆                    ┆                    │
│ 823577   ┆ 471 ┆ 725 ┆ 101 ┆ 45 nL        ┆ 2026-03-13 ┆ 823577_2026-03-13

In [26]:
%store vs200_annotations_df

Stored 'vs200_annotations_df' (DataFrame)


## Import and Clean Tissuecyte Injection Data

In [27]:
!uv sync --extra sdk -q
# restart kernel after running!

In [1]:
# from tissuecyte_injections_coords_utils import sort_structures_by_region
from tissuecyte_injections_coords_utils import load_csv_annotations
from tissuecyte_injections_coords_utils import get_structure_info_for_annotations

In [2]:
from tissuecyte_injections_coords_utils import annotation, meta 
from tissuecyte_injections_coords_utils import tree

In [3]:
import polars as pl
tissuecyte_annotations = load_csv_annotations(r"Z:\Vayle\Muscimol\injections_ccf_coordinates")
tissuecyte_annotations_df = pl.concat(
	[
		pl.from_pandas(df).with_columns(pl.lit(filename).alias("mouse_id"))
		for filename, df in tissuecyte_annotations.items()
	],
	how="vertical_relaxed",
)
tissuecyte_annotations_df = tissuecyte_annotations_df.drop(["Unnamed: 0"]).with_columns(
	pl.col("mouse_id").str.replace(r"\.csv$", "")
)

In [4]:
%store tissuecyte_annotations_df

Stored 'tissuecyte_annotations_df' (DataFrame)


## Combine VS200, Tissuecyte, and Neuroglancer dataframes

In [5]:
#restore dataframes from neuroglancer and vs200 annotations ran in dr env previously

%store -r neuroglancer_annotations_df
%store -r vs200_annotations_df

In [6]:
# Vertically concatenate both dataframes - matching columns get appended, unique columns filled with nulls
total_injections_df = pl.concat(
    [tissuecyte_annotations_df,neuroglancer_annotations_df, vs200_annotations_df],
    how="diagonal_relaxed"
)



In [7]:
total_injections_df

AP,DV,ML,session_id,mouse_id,description,date,target_brain_region
i64,i64,i64,str,str,str,str,str
252,405,675,"""728053_2024-11-19""","""728053""",null,null,null
260,400,460,"""728053_2024-11-19""","""728053""",null,null,null
475,300,430,"""728053_2024-11-21""","""728053""",null,null,null
437,332,710,"""728053_2024-11-21""","""728053""",null,null,null
877,235,480,"""737412_2024-12-19""","""737412""",null,null,null
…,…,…,…,…,…,…,…
503,91,396,"""856846_2026-06-05""","""856846""","""45 nL nonconugated muscimol""","""2026-06-05""","""Ant. Str_RH"""
305,174,596,"""856846_2026-06-09""","""856846""","""45 nL nonconugated muscimol""","""2026-06-09""","""ORBm_LH"""
295,168,536,"""856846_2026-06-09""","""856846""","""45 nL nonconugated muscimol""","""2026-06-09""","""ORBm_RH"""


## Align CCF structures to injection coordinates and injection descriptions for all annotated behavior sessions

In [8]:
# Get structure info and add as new column
structure_names = get_structure_info_for_annotations(total_injections_df, annotation, tree)
injections_df = total_injections_df.with_columns(
    pl.Series("ccf_structure", structure_names)
)

Coord: (252, 405, 675) | Structure: Anterior olfactory nucleus
Coord: (260, 400, 460) | Structure: Orbital area, ventrolateral part, layer 1
Coord: (475, 300, 430) | Structure: Caudoputamen
Coord: (437, 332, 710) | Structure: Caudoputamen
Coord: (877, 235, 480) | Structure: Superior colliculus, motor related, intermediate white layer
Coord: (875, 230, 620) | Structure: Superior colliculus, motor related, intermediate white layer
Coord: (305, 380, 470) | Structure: Orbital area, ventrolateral part, layer 5
Coord: (327, 370, 720) | Structure: Orbital area, lateral part, layer 5
Coord: (352, 367, 720) | Structure: Orbital area, lateral part, layer 6a
Coord: (297, 375, 467) | Structure: Orbital area, ventrolateral part, layer 5
Coord: (855, 202, 505) | Structure: Superior colliculus, motor related, intermediate gray layer
Coord: (857, 200, 655) | Structure: Superior colliculus, optic layer
Coord: (855, 202, 500) | Structure: Superior colliculus, optic layer
Coord: (850, 190, 657) | Structu

In [12]:
# Create dataframe with AP, DV, ML and CCF structure + add injection description and session id + target_brain_region
ccf_structures_aligned_to_session_df = total_injections_df.select(["AP", "DV", "ML", "description", "session_id", "target_brain_region"]).with_columns(
    pl.Series("CCF_Structure", structure_names)
)
 
ccf_structures_aligned_to_session_df.to_pandas().to_excel(r"Z:\Vayle\Muscimol\ccf_structures_aligned_to_session.xlsx")

In [13]:
print(ccf_structures_aligned_to_session_df)

shape: (105, 7)
┌─────┬─────┬─────┬──────────────┬───────────────────┬──────────────────────┬──────────────────────┐
│ AP  ┆ DV  ┆ ML  ┆ description  ┆ session_id        ┆ target_brain_region  ┆ CCF_Structure        │
│ --- ┆ --- ┆ --- ┆ ---          ┆ ---               ┆ ---                  ┆ ---                  │
│ i64 ┆ i64 ┆ i64 ┆ str          ┆ str               ┆ str                  ┆ str                  │
╞═════╪═════╪═════╪══════════════╪═══════════════════╪══════════════════════╪══════════════════════╡
│ 252 ┆ 405 ┆ 675 ┆ null         ┆ 728053_2024-11-19 ┆ null                 ┆ Anterior olfactory   │
│     ┆     ┆     ┆              ┆                   ┆                      ┆ nucleus              │
│ 260 ┆ 400 ┆ 460 ┆ null         ┆ 728053_2024-11-19 ┆ null                 ┆ Orbital area,        │
│     ┆     ┆     ┆              ┆                   ┆                      ┆ ventrolateral pa…    │
│ 475 ┆ 300 ┆ 430 ┆ null         ┆ 728053_2024-11-21 ┆ null                

## Concatenate trials and performance tables to CCF-aligned injections coordinates 

In [ ]:
import npc_sessions
from npc_sessions import DynamicRoutingSession

In [ ]:
session = '767405_2025-05-19'
session = npc_sessions.Session(session)

In [13]:
# Get trials and performance data for all sessions and join to injections_df
trials_list = []
performance_list = []

# Get unique session_ids from total_injections_df
session_ids = total_injections_df.select("session_id").drop_nulls().unique().to_series().to_list()

for session_id in session_ids:
    try:
        session = npc_sessions.Session(session_id)
        
        # Get trials table and add session_id
        trials = pl.from_pandas(session.trials[:]).with_columns(
            pl.lit(session_id).alias("session_id")
        )
        trials_list.append(trials)
        
        # Get performance table and add session_id
        performance = pl.from_pandas(session.performance[:]).with_columns(
            pl.lit(session_id).alias("session_id")
        )
        performance_list.append(performance)
        
        print(f"Loaded: {session_id}")
    except Exception as e:
        print(f"Failed to load {session_id}: {e}")

        # Combine all trials and performance data
if trials_list:
    all_trials_df = pl.concat(trials_list, how="diagonal_relaxed")
if performance_list:
    all_performance_df = pl.concat(performance_list, how="diagonal_relaxed")

print(f"\nLoaded {len(trials_list)} sessions")

# Join injections with trials (each injection row repeats for each trial)
injections_with_trials_df = total_injections_df.join(
    all_trials_df,
    on="session_id",
    how="left"
)

# Join injections with performance (summary metrics per session)
injections_with_performance_df = total_injections_df.join(
    all_performance_df,
    on="session_id", 
    how="left"
)

print(f"Injections with trials: {injections_with_trials_df.shape}")
print(f"Injections with performance: {injections_with_performance_df.shape}")

Failed to load 789937_2025-10-24: Unable to locate credentials
Failed to load 772657_2025-02-25: Unable to locate credentials
Failed to load 805749_2025-10-28: Unable to locate credentials
Failed to load 807743_2025-11-14: Unable to locate credentials
Failed to load 767405_2025-05-23: Unable to locate credentials
Failed to load 789937_2025-10-21: Unable to locate credentials
Failed to load 772657_2025-03-04: Unable to locate credentials
Failed to load 767405_2025-05-20: Unable to locate credentials
Failed to load 807742_2025-10-07: Unable to locate credentials
Failed to load 746138_2024-12-18: Unable to locate credentials
Failed to load 807743_2025-11-11: Unable to locate credentials
Failed to load 728053_2024-11-21: Unable to locate credentials
Failed to load 774470_2025-07-30: Unable to locate credentials
Failed to load 737412_2024-12-19: Unable to locate credentials
Failed to load 789937_2025-10-17: Unable to locate credentials
Failed to load 774916_2025-03-28: Unable to locate cred

KeyboardInterrupt: 

In [ ]:
injections_with_trials_df

In [ ]:
injections_with_performance_df

## CCF Strcutures with Performance

In [ ]:
pip install openpyxl

In [10]:
# Join CCF structure onto injections_with_performance_df based on AP, DV, ML
# Drop existing CCF_Structure columns to allow re-running this cell
cols_to_drop = [c for c in injections_with_performance_df.columns if "CCF_Structure" in c]
if cols_to_drop:
    injections_with_performance_df = injections_with_performance_df.drop(cols_to_drop)

injections_with_performance_df = injections_with_performance_df.join(
    ccf_structures_aligned_to_session,
    on=["AP", "DV", "ML"],
    how="left"
)
injections_with_performance_df

injections_with_performance_df.to_pandas().to_excel(r"Z:\Muscimol\ccf_labeled_injections_with_performance.xlsx")

NameError: name 'injections_with_performance_df' is not defined

In [ ]:
# Join CCF structure onto injections_with_performance_df based on AP, DV, ML
injections_with_performance_df = injections_with_performance_df.join(
    ccf_structures_aligned_to_session,
    on=["AP", "DV", "ML"],
    how="left"
)
injections_with_performance_df

injections_with_performance_df.to_pandas().to_excel(r"Z:\Muscimol\ccf_labeled_injections_with_performance.xlsx")

In [ ]:
# Compare mean visual target response rate by session_id and CCF structure
perf_df = injections_with_performance_df

# Resolve column names robustly
structure_col = "CCF_Structure" if "CCF_Structure" in perf_df.columns else "ccf_structure"
vis_target_col = next(
    c for c in perf_df.columns
    if ("vis_target" in c.lower()) and ("response_rate" in c.lower())
)

# Exclude striatum/midbrain-related structures
exclude_keywords = [
    "striatum", "caudoputamen", "nucleus accumbens",
    "midbrain", "superior colliculus", "substantia nigra", "ventral tegmental", "ventricle"
]

filtered = (
    perf_df
    .filter(
        pl.col(structure_col).is_not_null() &
        ~pl.col(structure_col).str.to_lowercase().str.contains("|".join(exclude_keywords))
    )
    .group_by(["session_id", structure_col])
    .agg(pl.col(vis_target_col).mean().alias("mean_vis_target_response_rate"))
)

# Pivot for heatmap: rows=session_id, cols=CCF structure
heatmap_df = (
    filtered
    .to_pandas()
    .pivot(index="session_id", columns=structure_col, values="mean_vis_target_response_rate")
    .sort_index()
)

plt.figure(figsize=(max(10, heatmap_df.shape[1] * 0.5), max(8, heatmap_df.shape[0] * 0.25)))
im = plt.imshow(heatmap_df.fillna(np.nan), aspect="auto", cmap="viridis", interpolation="none")
plt.colorbar(im, label="Mean vis target response rate")
plt.yticks(range(len(heatmap_df.index)), heatmap_df.index)
plt.xticks(range(len(heatmap_df.columns)), heatmap_df.columns, rotation=90)
plt.xlabel("CCF Structure (excluding striatum/midbrain)")
plt.ylabel("Session ID")
plt.title("Vis Target Response Rate by Session and CCF Structure")
plt.tight_layout()
plt.show()

## Plot some behavior metric versus ccf coord on annotation volume

In [ ]:
from tissuecyte_injections_coords_utils import reference_space_key
from tissuecyte_injections_coords_utils import rsp
from tissuecyte_injections_coords_utils import rspc
reference_space_key = "annotation/ccf_2022"

In [ ]:
%pip install plotly

In [ ]:
%pip install nbformat>=4.2.0

In [ ]:
# Plot coords in CCF space colored by vis target response rate -coronal
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Get the vis target response rate column
vis_target_col = next(
    c for c in injections_with_performance_df.columns
    if ("vis_target" in c.lower()) and ("response_rate" in c.lower())
)
structure_col = "CCF_Structure" if "CCF_Structure" in injections_with_performance_df.columns else "ccf_structure"

# Convert to pandas for plotting
plot_df = injections_with_performance_df.select(
    ["AP", "DV", "ML", "session_id", structure_col, vis_target_col]
).drop_nulls(subset=["AP", "DV", "ML", vis_target_col]).to_pandas()

# Exclude caudatoputamen, ventricle, and colliculus structures
exclude_keywords = ['caudoputamen', 'caudatoputamen', 'ventricle', 'colliculus']
mask = ~plot_df[structure_col].str.lower().str.contains('|'.join(exclude_keywords), na=True)
plot_df = plot_df[mask].copy()

# Add percentage column for display
plot_df['vis_target_pct'] = plot_df[vis_target_col] * 100

ap_index = 400  # AP slice index for coronal view

# Get coronal slice image from reference space
try:
    # Axis 0 = coronal (AP slicing direction)
    slice_image = rsp.get_slice_image(0, ap_index * 10)
    def rgb2gray(rgb):
        return np.dot(rgb[..., :3], [0.2989, 0.5870, 0.1140])
    gray_image = rgb2gray(slice_image)
    has_background = True
except Exception as e:
    print(f"Could not load coronal slice image: {e}")
    has_background = False

# Create figure with coronal slice background
fig = go.Figure()

# Add coronal slice background image
if has_background:
    fig.add_trace(go.Image(
        z=np.stack([gray_image, gray_image, gray_image], axis=-1).astype(np.uint8),
        x0=0,
        y0=0,
        dx=1140 / gray_image.shape[1],  # Scale to ML axis
        dy=800 / gray_image.shape[0],   # Scale to DV axis
        opacity=0.6
    ))

# Add scatter plot on top
fig.add_trace(go.Scatter(
    x=plot_df['ML'],
    y=plot_df['DV'],
    mode='markers',
    marker=dict(
        size=12,
        color=plot_df['vis_target_pct'],
        colorscale='RdBu_r',
        cmin=0,
        cmax=100,
        line=dict(width=1, color='black'),
        colorbar=dict(title='Vis Target<br>Response Rate (%)')
    ),
    customdata=plot_df[['session_id', structure_col, 'ML', 'DV', 'AP', 'vis_target_pct']].values,
    hovertemplate='<b>Session:</b> %{customdata[0]}<br>' +
                  '<b>Structure:</b> %{customdata[1]}<br>' +
                  '<b>ML:</b> %{x:.0f} μm<br>' +
                  '<b>DV:</b> %{y:.0f} μm<br>' +
                  '<b>AP:</b> %{customdata[4]:.0f} μm<br>' +
                  '<b>Response Rate:</b> %{customdata[5]:.1f}%<extra></extra>'
))

# Update layout
fig.update_layout(
    title=f'CCF Coronal Injection Sites Colored by Vis Target Response Rate',
    xaxis=dict(title='ML', range=[0, 1140]),
    yaxis=dict(title='DV', range=[800, 0], scaleanchor='x'),  # Invert y-axis
    width=900,
    height=700,
    showlegend=False
)

fig.show()

In [ ]:
# Plot coords in CCF space colored by vis target response rate - sagittal
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from PIL import Image

# Get the vis target response rate column
vis_target_col = next(
    c for c in injections_with_performance_df.columns
    if ("vis_target" in c.lower()) and ("response_rate" in c.lower())
)
structure_col = "CCF_Structure" if "CCF_Structure" in injections_with_performance_df.columns else "ccf_structure"

# Convert to pandas for plotting
plot_df = injections_with_performance_df.select(
    ["AP", "DV", "ML", "session_id", structure_col, vis_target_col]
).drop_nulls(subset=["AP", "DV", "ML", vis_target_col]).to_pandas()

# Exclude caudatoputamen, ventricle, and colliculus structures
exclude_keywords = ['caudoputamen', 'caudatoputamen', 'ventricle', 'colliculus']
mask = ~plot_df[structure_col].str.lower().str.contains('|'.join(exclude_keywords), na=True)
plot_df = plot_df[mask].copy()

# Add percentage column for display
plot_df['vis_target_pct'] = plot_df[vis_target_col] * 100

# Use median ML of the data points for the slice
ml_index = int(plot_df['ML'].median())
print(f"Using ML slice at {ml_index*10} μm (median of injection points)")
print(f"AP range of points: {plot_df['AP'].min()*10} - {plot_df['AP'].max()*10} μm")
print(f"DV range of points: {plot_df['DV'].min()*10} - {plot_df['DV'].max()*10} μm")

# Get sagittal slice image from reference space
try:
    # Axis 2 = sagittal (ML slicing direction)
    slice_image = rsp.get_slice_image(2, ml_index * 10)
    def rgb2gray(rgb):
        return np.dot(rgb[..., :3], [0.2989, 0.5870, 0.1140])
    gray_image = rgb2gray(slice_image)
    # Transpose to match matplotlib's imshow with .T and extent order
    gray_image = gray_image.T
    has_background = True
except Exception as e:
    print(f"Could not load sagittal slice image: {e}")
    has_background = False

# Create figure with sagittal slice background
fig = go.Figure()

# Add sagittal slice background image
if has_background:
    # Convert to RGB uint8 and create PIL Image
    img_rgb = np.stack([gray_image, gray_image, gray_image], axis=-1).astype(np.uint8)
    pil_img = Image.fromarray(img_rgb)
    
    # Match matplotlib's extent=[1320, 0, 800, 0] behavior
    # Image goes from x=1320 (left edge) to x=0 (right edge), y=800 (bottom) to y=0 (top)
    fig.add_layout_image(
        dict(
            source=pil_img,
            xref="x",
            yref="y",
            x=1320,  # Left edge of image
            y=0,     # Top edge of image (since y-axis is inverted)
            sizex=1320,
            sizey=800,
            xanchor="left",
            yanchor="top",
            sizing="stretch",
            opacity=0.6,
            layer="below"
        )
    )

# Add scatter plot on top - flip AP for proper orientation (matching matplotlib: 1320 - AP)
fig.add_trace(go.Scatter(
    x=1320 - plot_df['AP'],  # Flip AP for sagittal orientation
    y=plot_df['DV'],
    mode='markers',
    marker=dict(
        size=12,
        color=plot_df['vis_target_pct'],
        colorscale='RdBu_r',
        cmin=0,
        cmax=100,
        line=dict(width=1, color='black'),
        colorbar=dict(title='Vis Target<br>Response Rate (%)')
    ),
    customdata=plot_df[['session_id', structure_col, 'ML', 'DV', 'AP', 'vis_target_pct']].values,
    hovertemplate='<b>Session:</b> %{customdata[0]}<br>' +
                  '<b>Structure:</b> %{customdata[1]}<br>' +
                  '<b>ML:</b> %{customdata[2]:.0f} μm<br>' +
                  '<b>DV:</b> %{y:.0f} μm<br>' +
                  '<b>AP:</b> %{customdata[4]:.0f} μm<br>' +
                  '<b>Response Rate:</b> %{customdata[5]:.1f}%<extra></extra>'
))

# Update layout - reverse x-axis to match matplotlib extent=[1320, 0, ...]
fig.update_layout(
    title=f'CCF Sagittal View Injection Sites Colored by Vis Target Response Rate',
    xaxis=dict(title='AP', range=[1320, 0], autorange=False),  # Reversed to match matplotlib
    yaxis=dict(title='DV', range=[800, 0], scaleanchor='x'),   # Inverted y-axis
    width=1100,
    height=700,
    showlegend=False
)

fig.show()